In [3]:
from pathlib import Path
import sys
import os

%load_ext autoreload
%autoreload 2

dir = Path().resolve().parents[1]

if dir not in sys.path:
    print("directory path is not in the system path")
    sys.path.append(str(dir))
    print("adding directory...")
else:
    print("Directory already exists in the system path")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
directory path is not in the system path
adding directory...


In [4]:
from nn import Unet1D, Returns, RMSELoss
from scripts import train
from utils import log_transform
import torch
from torch.utils.data import DataLoader, random_split
import yfinance as yf
import pandas as pd
import numpy as np
import math

[10, 20]
[10, 20, 40, 30]


In [5]:
ticker = "^GSPC"
start_interval = "2016-12-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = torch.tensor(yf.Ticker(ticker).history(start=start_interval, end=end_interval, interval=interval)["Close"].to_numpy())

In [6]:
split_idx = len(raw_snp500) - math.ceil(len(raw_snp500) * 0.2)
train_raw_snp500, test_raw_snp500 = raw_snp500[:split_idx], raw_snp500[split_idx:]

window_size = 32

train_data = Returns(
  raw_returns=train_raw_snp500,
  window_size=window_size,
  transform=log_transform
)
test_data = Returns(
  raw_returns=test_raw_snp500,
  window_size=window_size,
  transform=log_transform
)

len(train_data), len(test_data)

(1794, 425)

In [7]:
train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=True)

next(iter(train_dataloader)).dtype

torch.float32

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device

'cpu'

In [9]:
betas = torch.linspace(1e-4, 2e-2, 1000)
type(1 - betas)

torch.Tensor

In [68]:
encoder_in_channels = [1, 4, 8, 16]
encoder_out_channels = [4, 8, 16, 32]
decoder_in_channels = [32, 16, 8, 4]
decoder_out_channels = [16, 8, 4, 1]
attn_res = 16
n_res_block = 2
T = 1000
num_heads = 4
betas = torch.linspace(1e-4, 2e-2, T)
alpha_hats = torch.cumprod(
  input=1-betas,
  dim=0,
  dtype=torch.float32
)

model = Unet1D(
  attn_res=attn_res,
  n_res_block=n_res_block,
  encoder_in_channels=encoder_in_channels,
  encoder_out_channels=encoder_out_channels,
  decoder_in_channels=decoder_in_channels,
  decoder_out_channels=decoder_out_channels,
  T=T,
  num_heads=num_heads
)

In [69]:
print(alpha_hats)
print(alpha_hats.size())

tensor([9.9990e-01, 9.9978e-01, 9.9964e-01, 9.9948e-01, 9.9930e-01, 9.9910e-01,
        9.9888e-01, 9.9864e-01, 9.9838e-01, 9.9811e-01, 9.9781e-01, 9.9749e-01,
        9.9715e-01, 9.9679e-01, 9.9641e-01, 9.9602e-01, 9.9560e-01, 9.9516e-01,
        9.9471e-01, 9.9423e-01, 9.9374e-01, 9.9322e-01, 9.9269e-01, 9.9213e-01,
        9.9156e-01, 9.9097e-01, 9.9035e-01, 9.8972e-01, 9.8907e-01, 9.8840e-01,
        9.8771e-01, 9.8700e-01, 9.8627e-01, 9.8553e-01, 9.8476e-01, 9.8398e-01,
        9.8317e-01, 9.8235e-01, 9.8151e-01, 9.8065e-01, 9.7977e-01, 9.7887e-01,
        9.7795e-01, 9.7702e-01, 9.7606e-01, 9.7509e-01, 9.7410e-01, 9.7309e-01,
        9.7206e-01, 9.7102e-01, 9.6995e-01, 9.6887e-01, 9.6777e-01, 9.6665e-01,
        9.6551e-01, 9.6436e-01, 9.6319e-01, 9.6200e-01, 9.6079e-01, 9.5956e-01,
        9.5832e-01, 9.5706e-01, 9.5578e-01, 9.5449e-01, 9.5318e-01, 9.5185e-01,
        9.5050e-01, 9.4914e-01, 9.4776e-01, 9.4636e-01, 9.4494e-01, 9.4351e-01,
        9.4207e-01, 9.4060e-01, 9.3912e-

In [70]:
loss_fn = RMSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [71]:
print(device)
train(
  train_data=train_dataloader,
  test_data=test_dataloader,
  optimizer=optimizer,
  loss_fn=loss_fn,
  epochs=10,
  alpha_hats=alpha_hats,
  model=model,
  scheduler=None,
  early_stopping=None,
  device=device,
  T=T
)

cpu


  0%|          | 0/10 [00:00<?, ?it/s]

start
X size before :  torch.Size([32, 1, 32])
batch_size :  32
t :  tensor([116, 467, 501, 152, 580, 647, 811, 289, 878, 119,  78,  29, 833, 370,
        301, 910, 896, 424,  55, 308, 372, 469, 861, 529, 614, 468, 144, 444,
        318, 283, 946, 819])
t size :  torch.Size([32])
xt :  tensor([[[-0.4093, -1.1669,  1.4640,  ..., -1.3471, -0.9239,  1.4332]],

        [[ 0.1513, -0.0255, -0.4700,  ..., -0.6733,  2.1982, -0.9277]],

        [[ 0.4474,  0.1152, -0.8432,  ...,  0.9666, -1.1766,  0.5349]],

        ...,

        [[ 0.4631, -0.2449, -0.0492,  ...,  1.2315, -1.6457,  0.0306]],

        [[-0.3350,  0.0729, -0.2593,  ..., -1.7254,  1.3346,  0.9487]],

        [[ 0.5231, -0.1640,  0.1401,  ...,  0.1352, -0.0216, -0.5952]]])
epsilon :  tensor([[[-1.1360, -1.2335,  1.5273,  ..., -1.7915, -0.9240,  1.4338]],

        [[ 0.4060, -0.0267, -0.4891,  ..., -0.9004,  2.1983, -0.9280]],

        [[ 1.2276,  0.1230, -0.8781,  ...,  1.2869, -1.1765,  0.5355]],

        ...,

        [[ 1.2598

  0%|          | 0/10 [00:03<?, ?it/s]

passed
X size :  torch.Size([32, 1, 32])
start
X size before :  torch.Size([32, 1, 32])
batch_size :  32
t :  tensor([212, 124,   0, 421, 814, 191, 464, 977, 971, 519, 243, 476, 782, 487,
         97, 454, 207,  77, 753,  81, 311, 833, 518, 367, 128, 575, 277, 230,
        467, 550,  54, 512])
t size :  torch.Size([32])
xt :  tensor([[[ 6.1608e-01, -4.0212e-02,  1.3154e-02,  ...,  2.7791e+00,
           8.1143e-02,  1.0667e+00]],

        [[ 8.2319e-01,  4.5910e-01,  1.6016e-02,  ...,  1.0170e+00,
          -9.1895e-02, -6.4217e-01]],

        [[-1.8485e-01, -2.1262e-01, -2.2921e-03,  ...,  1.5814e+00,
           2.0411e-01,  6.1958e-01]],

        ...,

        [[ 7.9801e-02, -8.3571e-01, -2.9548e-03,  ...,  4.7655e-01,
          -4.8843e-02,  9.5509e-01]],

        [[ 8.4535e-01, -5.0948e-03,  2.6537e-03,  ...,  1.5955e+00,
           3.1089e-01, -1.3026e-01]],

        [[ 2.2811e-01, -3.1736e-01,  1.2071e-02,  ..., -1.1468e+00,
           4.2641e-02,  2.2371e-01]]])
epsilon :  tenso

RuntimeError: The size of tensor a (2) must match the size of tensor b (32) at non-singleton dimension 2